<center>

# Mathematical Model

</center>

Sets and parameters:
- $I$: set of demand points
- $J$: set of candidate facility points
- $d_{ij}$: distance between damand point i and candidate facility point j
- $R$: coverage radius (in meters)
- $a_{ij}$: Coverage indicator. 1 if $ d_{ij} \le R$, 0 otherwise, where $\quad a_{ij} \in \{0,1\} $
- $w_{i}$: Demand weight of grid i
- $W_{tot}$: Total weighted Demand
- $\tau$: Target coverage ratio, where $\quad \tau \in (0,1]$


Decision variables:
- $x_j$: 1 if facility j is selected, 0 otherwise, where $x_j \in (0,1)$
- $z_{i}$: 1 if demand point i is covered (within radius R), 0 otherwise, where $z_i \in (0,1)$



$$
\begin{align}
    \min \quad & \sum_{j \in J} w_id_{ij}y_{ij}  \\
    \text{s.t.} \quad 
    & \sum_{j \in J} a_{ij}x_j \ge z_i \quad \forall i \in I\\
    & \sum_{i \in I} w_iz_i \ge \tau W_{tot}\\
    & x_j \in \{0,1\}, z_j \in \{0,1\}
\end{align}
$$

In [ ]:
# -*- coding: utf-8 -*-
import geopandas as gpd
import pandas as pd, numpy as np
from shapely.ops import unary_union
import gurobipy as gp
from gurobipy import GRB

# --------- 입력 ----------
daejeok_grid_shp = r"/Users/anjunho/Desktop/물류 데이터 공모전/인구밀도_대덕구.shp"
transit_csv      = r"/Users/anjunho/Desktop/물류 데이터 공모전/preprocessing ver7.csv"
cands_csv        = r"/Users/anjunho/Desktop/물류 데이터 공모전/finaldestination.csv"
R_COVER_M        = 1500        # 커버 반경 (m)
TARGET_COVER     = 0.90         # 목표 커버리지(가중치 기준 90%)

POP_W, TRANSIT_W = 0.7, 0.3     # 총인구수 : 유동인구 가중치 7:3

# --------- 1) 데이터 준비: 대덕구 격자/가중치 w_i ----------
gdf = gpd.read_file(daejeok_grid_shp)
gdf = gdf[gdf.geometry.notnull() & gdf.geometry.is_valid].copy()
if (not gdf.crs) or (not gdf.crs.is_projected):
    gdf = gdf.to_crs(5179)

pop_col = "val"
gdf[pop_col] = pd.to_numeric(gdf[pop_col], errors="coerce").fillna(0)

df_pts = pd.read_csv(transit_csv)
df_pts = df_pts[df_pts["시군구명"].astype(str).str.contains("대덕구", na=False)].copy()
gdf_pts = gpd.GeoDataFrame(df_pts,
            geometry=gpd.points_from_xy(df_pts["경도"], df_pts["위도"]),
            crs=4326).to_crs(5179)

joined = gpd.sjoin(gdf_pts, gdf[["geometry"]], how="left", predicate="intersects")
joined["월 평균 승하차 인원"] = pd.to_numeric(joined["월 평균 승하차 인원"], errors="coerce").fillna(0)
bus_mean = joined[joined["시설 종류"]=="버스"]   .groupby("index_right")["월 평균 승하차 인원"].mean()
sub_mean = joined[joined["시설 종류"]=="지하철"].groupby("index_right")["월 평균 승하차 인원"].mean()

gdf["bus_mean"] = gdf.index.map(bus_mean).fillna(0)
gdf["sub_mean"] = gdf.index.map(sub_mean).fillna(0)
gdf["transit"]  = gdf["bus_mean"] + gdf["sub_mean"]

# Min–Max → w_i
def minmax(s):
    s = s.astype(float)
    lo, hi = s.min(), s.max()
    return pd.Series(0.0, s.index) if hi-lo < 1e-12 else (s-lo)/(hi-lo)

gdf["pop_n"] = minmax(gdf[pop_col])
gdf["tr_n"]  = minmax(gdf["transit"])
gdf["w"]     = POP_W*gdf["pop_n"] + TRANSIT_W*gdf["tr_n"]

# 수요점은 격자 중심
gdf_cent = gdf.copy()
gdf_cent["geometry"] = gdf_cent.geometry.centroid
I = list(range(len(gdf_cent)))

# --------- 2) 후보지 준비 ----------
df_c = pd.read_csv(cands_csv, encoding="utf-8-sig")
gdf_c = gpd.GeoDataFrame(df_c,
         geometry=gpd.points_from_xy(df_c["경도"], df_c["위도"]),
         crs=4326).to_crs(5179)
J = list(range(len(gdf_c)))

# --------- 3) 커버 행렬 a_ij (within R_COVER_M) ----------
ci = np.array([(p.x, p.y) for p in gdf_cent.geometry])
cj = np.array([(p.x, p.y) for p in gdf_c.geometry])
dmat = np.sqrt(((ci[:,None,:] - cj[None,:,:])**2).sum(axis=2))   # |I|x|J|
A = (dmat <= R_COVER_M).astype(int)  # a_ij

# --------- 4) 최소 p로 목표 커버리지 달성 (maximize covered, then binary search p or minimize Sum x) ----------
# 여기서는 '최소 p'를 직접 최적화: Minimize Sum x, with weighted coverage constraint
Wtot = float(gdf_cent["w"].sum())

m = gp.Model("min_p_for_target_coverage")
x = m.addVars(J, vtype=GRB.BINARY, name="x")          # 후보 설치 여부
z = m.addVars(I, vtype=GRB.BINARY, name="z")          # 수요 i가 커버되면 1

# 커버 논리: sum_j a_ij x_j >= z_i
for i in I:
    m.addConstr(gp.quicksum(A[i,j]*x[j] for j in J) >= z[i])

# 목표 커버리지 (가중치 기준)
m.addConstr(gp.quicksum(gdf_cent.loc[i,"w"] * z[i] for i in I) >= TARGET_COVER * Wtot)

# 목표: 설치 개수 최소화
m.setObjective(gp.quicksum(x[j] for j in J), GRB.MINIMIZE)
m.Params.TimeLimit = 300
m.optimize()

open_sites = [j for j in J if x[j].X > 0.5]
p_min = len(open_sites)
covered_weight = sum(gdf_cent.loc[i,"w"] for i in I if z[i].X > 0.5)
print(f"[RESULT] 최소 설치 개수 p = {p_min} (가중 커버리지 = {covered_weight/Wtot:.1%}, R={R_COVER_M}m)")

Set parameter TimeLimit to value 300
Gurobi Optimizer version 12.0.3 build v12.0.3rc0 (mac64[arm] - Darwin 25.0.0 25A362)

CPU model: Apple M1 Pro
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Non-default parameters:
TimeLimit  300

Optimize a model with 344 rows, 444 columns and 1674 nonzeros
Model fingerprint: 0xb6054b2f
Variable types: 0 continuous, 444 integer (444 binary)
Coefficient statistics:
  Matrix range     [3e-04, 1e+00]
  Objective range  [1e+00, 1e+00]
  Bounds range     [1e+00, 1e+00]
  RHS range        [2e+01, 2e+01]
Found heuristic solution: objective 11.0000000
Presolve removed 248 rows and 308 columns
Presolve time: 0.00s
Presolved: 96 rows, 136 columns, 1011 nonzeros
Found heuristic solution: objective 10.0000000
Variable types: 0 continuous, 136 integer (136 binary)
Found heuristic solution: objective 9.0000000

Root relaxation: objective 3.565619e+00, 84 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current Node    |

In [ ]:
# -*- coding: utf-8 -*-
import geopandas as gpd
import pandas as pd, numpy as np
from shapely.ops import unary_union
import gurobipy as gp
from gurobipy import GRB

# --------- 입력 ----------
daejeok_grid_shp = r"/Users/anjunho/Desktop/물류 데이터 공모전/인구밀도_서구.shp"
transit_csv      = r"/Users/anjunho/Desktop/물류 데이터 공모전/preprocessing ver7.csv"
cands_csv        = r"/Users/anjunho/Desktop/물류 데이터 공모전/finaldestination.csv"
R_COVER_M        = 1500        # 커버 반경 (m)
TARGET_COVER     = 0.90         # 목표 커버리지(가중치 기준 90%)

POP_W, TRANSIT_W = 0.7, 0.3     # 총인구수 : 유동인구 가중치 7:3

# --------- 1) 데이터 준비: 대덕구 격자/가중치 w_i ----------
gdf = gpd.read_file(daejeok_grid_shp)
gdf = gdf[gdf.geometry.notnull() & gdf.geometry.is_valid].copy()
if (not gdf.crs) or (not gdf.crs.is_projected):
    gdf = gdf.to_crs(5179)

pop_col = "val"
gdf[pop_col] = pd.to_numeric(gdf[pop_col], errors="coerce").fillna(0)

df_pts = pd.read_csv(transit_csv)
df_pts = df_pts[df_pts["시군구명"].astype(str).str.contains("대덕구", na=False)].copy()
gdf_pts = gpd.GeoDataFrame(df_pts,
            geometry=gpd.points_from_xy(df_pts["경도"], df_pts["위도"]),
            crs=4326).to_crs(5179)

joined = gpd.sjoin(gdf_pts, gdf[["geometry"]], how="left", predicate="intersects")
joined["월 평균 승하차 인원"] = pd.to_numeric(joined["월 평균 승하차 인원"], errors="coerce").fillna(0)
bus_mean = joined[joined["시설 종류"]=="버스"]   .groupby("index_right")["월 평균 승하차 인원"].mean()
sub_mean = joined[joined["시설 종류"]=="지하철"].groupby("index_right")["월 평균 승하차 인원"].mean()

gdf["bus_mean"] = gdf.index.map(bus_mean).fillna(0)
gdf["sub_mean"] = gdf.index.map(sub_mean).fillna(0)
gdf["transit"]  = gdf["bus_mean"] + gdf["sub_mean"]

# Min–Max → w_i
def minmax(s):
    s = s.astype(float)
    lo, hi = s.min(), s.max()
    return pd.Series(0.0, s.index) if hi-lo < 1e-12 else (s-lo)/(hi-lo)

gdf["pop_n"] = minmax(gdf[pop_col])
gdf["tr_n"]  = minmax(gdf["transit"])
gdf["w"]     = POP_W*gdf["pop_n"] + TRANSIT_W*gdf["tr_n"]

# 수요점은 격자 중심
gdf_cent = gdf.copy()
gdf_cent["geometry"] = gdf_cent.geometry.centroid
I = list(range(len(gdf_cent)))

# --------- 2) 후보지 준비 ----------
df_c = pd.read_csv(cands_csv, encoding="utf-8-sig")
gdf_c = gpd.GeoDataFrame(df_c,
         geometry=gpd.points_from_xy(df_c["경도"], df_c["위도"]),
         crs=4326).to_crs(5179)
J = list(range(len(gdf_c)))

# --------- 3) 커버 행렬 a_ij (within R_COVER_M) ----------
ci = np.array([(p.x, p.y) for p in gdf_cent.geometry])
cj = np.array([(p.x, p.y) for p in gdf_c.geometry])
dmat = np.sqrt(((ci[:,None,:] - cj[None,:,:])**2).sum(axis=2))   # |I|x|J|
A = (dmat <= R_COVER_M).astype(int)  # a_ij

# --------- 4) 최소 p로 목표 커버리지 달성 (maximize covered, then binary search p or minimize Sum x) ----------
# 여기서는 '최소 p'를 직접 최적화: Minimize Sum x, with weighted coverage constraint
Wtot = float(gdf_cent["w"].sum())

m = gp.Model("min_p_for_target_coverage")
x = m.addVars(J, vtype=GRB.BINARY, name="x")          # 후보 설치 여부
z = m.addVars(I, vtype=GRB.BINARY, name="z")          # 수요 i가 커버되면 1

# 커버 논리: sum_j a_ij x_j >= z_i
for i in I:
    m.addConstr(gp.quicksum(A[i,j]*x[j] for j in J) >= z[i])

# 목표 커버리지 (가중치 기준)
m.addConstr(gp.quicksum(gdf_cent.loc[i,"w"] * z[i] for i in I) >= TARGET_COVER * Wtot)

# 목표: 설치 개수 최소화
m.setObjective(gp.quicksum(x[j] for j in J), GRB.MINIMIZE)
m.Params.TimeLimit = 300
m.optimize()

open_sites = [j for j in J if x[j].X > 0.5]
p_min = len(open_sites)
covered_weight = sum(gdf_cent.loc[i,"w"] for i in I if z[i].X > 0.5)
print(f"[RESULT] 최소 설치 개수 p = {p_min} (가중 커버리지 = {covered_weight/Wtot:.1%}, R={R_COVER_M}m)")

Set parameter TimeLimit to value 300
Gurobi Optimizer version 12.0.3 build v12.0.3rc0 (mac64[arm] - Darwin 25.0.0 25A362)

CPU model: Apple M1 Pro
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Non-default parameters:
TimeLimit  300

Optimize a model with 459 rows, 559 columns and 2164 nonzeros
Model fingerprint: 0xf4e09080
Variable types: 0 continuous, 559 integer (559 binary)
Coefficient statistics:
  Matrix range     [4e-04, 1e+00]
  Objective range  [1e+00, 1e+00]
  Bounds range     [1e+00, 1e+00]
  RHS range        [3e+01, 3e+01]
Found heuristic solution: objective 14.0000000
Presolve removed 308 rows and 356 columns
Presolve time: 0.00s
Presolved: 151 rows, 203 columns, 1432 nonzeros
Found heuristic solution: objective 13.0000000
Variable types: 0 continuous, 203 integer (203 binary)
Found heuristic solution: objective 12.0000000

Root relaxation: objective 5.267443e+00, 162 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current Node  